In [2]:
from pathlib import Path
import pandas as pd

here = Path.cwd()
ROOT = next(p for p in [here, *here.parents]  if (p / "data" / "raw").exists())
df  = pd.read_csv(ROOT / "data" / "raw" / "DataCoSupplyChainDataset.csv", encoding="latin-1")
pd.set_option("display.max_columns", 60)
df.head(3)

,Type,Days for shipping (real),Days for shipment (scheduled),Benefit per order,Sales per customer,Delivery Status,Late_delivery_risk,Category Id,Category Name,Customer City,Customer Country,Customer Email,Customer Fname,Customer Id,Customer Lname,Customer Password,Customer Segment,Customer State,Customer Street,Customer Zipcode,Department Id,Department Name,Latitude,Longitude,Market,Order City,Order Country,Order Customer Id,order date (DateOrders),Order Id,Order Item Cardprod Id,Order Item Discount,Order Item Discount Rate,Order Item Id,Order Item Product Price,Order Item Profit Ratio,Order Item Quantity,Sales,Order Item Total,Order Profit Per Order,Order Region,Order State,Order Status,Order Zipcode,Product Card Id,Product Category Id,Product Description,Product Image,Product Name,Product Price,Product Status,shipping date (DateOrders),Shipping Mode
0,DEBIT,3,4,91.250000,314.640015,Advance shipping,0,73,Sporting Goods,Caguas,Puerto Rico,XXXXXXXXX,Cally,20755,Holloway,XXXXXXXXX,Consumer,PR,5365 Noble Nectar Island,725.0,2,Fitness,18.251453,-66.037056,Pacific Asia,Bekasi,Indonesia,20755,1/31/2018 22:56,77202,1360,13.110000,0.04,180517,327.75,0.29,1,327.75,314.640015,91.250000,Southeast Asia,Java Occidental,COMPLETE,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,2/3/2018 22:56,Standard Class
1,TRANSFER,5,4,-249.089996,311.359985,Late delivery,1,73,Sporting Goods,Caguas,Puerto Rico,XXXXXXXXX,Irene,19492,Luna,XXXXXXXXX,Consumer,PR,2679 Rustic Loop,725.0,2,Fitness,18.279451,-66.037064,Pacific Asia,Bikaner,India,19492,1/13/2018 12:27,75939,1360,16.389999,0.05,179254,327.75,-0.80,1,327.75,311.359985,-249.089996,South Asia,Rajastán,PENDING,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,1/18/2018 12:27,Standard Class
2,CASH,4,4,-247.779999,309.720001,Shipping on time,0,73,Sporting Goods,San Jose,EE. UU.,XXXXXXXXX,Gillian,19491,Maldonado,XXXXXXXXX,Consumer,CA,8510 Round Bear Gate,95125.0,2,Fitness,37.292233,-121.881279,Pacific Asia,Bikaner,India,19491,1/13/2018 12:06,75938,1360,18.030001,0.06,179253,327.75,-0.80,1,327.75,309.720001,-247.779999,South Asia,Rajastán,CLOSED,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,1/17/2018 12:06,Standard Class


In [3]:
print("rows:", len(df))
print("unique orders:", df["Order Id"].nunique())
print("avg rows per order:", round(len(df)// df["Order Id"].nunique(), 2))
print("does lateness vary within an order?", (df.groupby("Order Id")["Late_delivery_risk"].nunique() > 1).any())

rows: 180519
unique orders: 65752
avg rows per order: 2
does lateness vary within an order? False


In [4]:
missing = df.isna().sum()
print(missing[missing > 0])
print()
print(df.dtypes.value_counts())

Customer Lname              8
Customer Zipcode            3
Order Zipcode          155679
Product Description    180519
dtype: int64

str        24
float64    15
int64      14
Name: count, dtype: int64


In [5]:
print(df["Late_delivery_risk"].value_counts(normalize=True).round(3))
print()
(df.groupby("Shipping Mode")
   .agg(rows=("Late_delivery_risk", "size"),
        late_rate=("Late_delivery_risk", "mean"),
        scheduled_days=("Days for shipment (scheduled)", "mean"),
        real_days=("Days for shipping (real)", "mean"))
   .round(3)
   .sort_values("late_rate", ascending=False))

Late_delivery_risk
1    0.548
0    0.452
Name: proportion, dtype: float64



,rows,late_rate,scheduled_days,real_days
Shipping Mode,,,,
First Class,27814,0.953,1.0,2.000
Second Class,35216,0.766,2.0,3.991
Same Day,9737,0.457,0.0,0.478
Standard Class,107752,0.381,4.0,3.996


In [6]:
print(pd.crosstab(df["Delivery Status"], df["Late_delivery_risk"]))
print()
real_over = (df["Days for shipping (real)"] > df["Days for shipment (scheduled)"]).astype(int)
print("late flag == (real days > scheduled days):", round((real_over == df["Late_delivery_risk"]).mean(), 4))
print()
print(pd.crosstab(df["Order Status"], df["Late_delivery_risk"]))

Late_delivery_risk      0      1
Delivery Status                 
Advance shipping    41592      0
Late delivery           0  98977
Shipping canceled    7754      0
Shipping on time    32196      0

late flag == (real days > scheduled days): 0.9755

Late_delivery_risk      0      1
Order Status                    
CANCELED             3692      0
CLOSED               8507  11109
COMPLETE            25292  34199
ON_HOLD              4354   5450
PAYMENT_REVIEW        811   1082
PENDING              8515  11712
PENDING_PAYMENT     16910  22922
PROCESSING           9399  12503
SUSPECTED_FRAUD      4062      0


In [7]:
df["order_date"] = pd.to_datetime(df["order date (DateOrders)"])
print(df["order_date"].min(), "→", df["order_date"].max())
df.set_index("order_date").resample("QE")["Late_delivery_risk"].agg(["size", "mean"]).round(3)

2015-01-01 00:00:00 → 2018-01-31 23:38:00


,size,mean
order_date,,
2015-03-31,15413,0.546
2015-06-30,15617,0.544
2015-09-30,15712,0.559
2015-12-31,15908,0.546
2016-03-31,15421,0.552
2016-06-30,15453,0.559
2016-09-30,15799,0.541
2016-12-31,15877,0.552
2017-03-31,15470,0.545
